In [4]:
import warnings
warnings.simplefilter("ignore", FutureWarning)

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# LightGBM
from lightgbm import LGBMClassifier

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    RandomizedSearchCV, StratifiedKFold
)

from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)

from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import randint, uniform, loguniform

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *

import src.preprocess_utils_diab14 as new_utils
sys.modules['src.preprocess_utils_diab'] = new_utils


print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 11:37:27


## 1. Load Data & Pipeline

In [5]:
BASE = Path.cwd().parent

PP2 = joblib.load(BASE/'src'/'preprocess_diabetes_v1.2.joblib')['preprocessador']

DATA_DIR = BASE/"data"/"raw"
X_train = pd.read_csv(DATA_DIR/"X_train_raw.csv")
X_test  = pd.read_csv(DATA_DIR/"X_test_raw.csv")
y_train = pd.read_csv(DATA_DIR/"y_train_raw.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR/"y_test_raw.csv").values.ravel()
mtd_scoring='roc_auc'

## 2.Baseline

In [8]:
# MODEL BASE
print("#Processo iniciado em:", time.strftime("%H:%M:%S"))
model_base = LGBMClassifier(objective="binary",n_jobs=-1,random_state=42,verbose=-1,
                            force_row_wise=True)

pipe_base  = pipe_models(model_base, PP2)
pipe_base.fit(X_train, y_train)

# =====================================================
# 1) Predição
# =====================================================
y_pred_base=pipe_base.predict(X_test)

# =====================================================
# 2) Otimização do threshold de decisão
# =====================================================
# Como classificadores probabilísticos usam threshold padrão de 0.5,
# testamos diferentes valores para verificar se existe um ponto que
# maximize a acurácia no conjunto de teste.
best_th_base,max_acc_base,y_probs_base=best_threshold2(pipe_base, X_train, y_train,X_test,y_test)

# =====================================================
# 3) Desempenho em validação cruzada
# =====================================================
# A validação cruzada utiliza a mesma métrica definida
# no processo de tuning para avaliar a estabilidade do modelo.
cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores_base = cross_val_score(pipe_base, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1)

print(f"\n{'='*70}")
print(f" 📍 RESULTADOS BASELINE".center(70))
print(f"{'='*70}")

# =====================================================
# 4) Avaliação por validação cruzada (Treino)
# =====================================================

print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_base.mean():>15.4f} ± {cv_scores_base.std():.4f}")

# =====================================================
# 5) Avaliação no conjunto de teste
# =====================================================
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_base):>10.4f}")
print(f"   Otimizado:                 {max_acc_base:>10.4f} (threshold ={best_th_base:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_base):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_base):>10.4f}")

# =====================================================
# 6) Hiperparametros utilizados
# =====================================================
print(f"{'='*70}")
print_hiper(model_base.get_params())
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))


#Processo iniciado em: 11:41:46

                         📍 RESULTADOS BASELINE                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7209 ± 0.0013

✅ TEST SET
   Padrão (0.5):                  0.6804
   Otimizado:                     0.6803 (threshold = 0.520)
   ROC-AUC:                       0.7211
   Avg precision:                 0.8073
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Boosting Type             : gbdt
• Class Weight              : None
• Colsample Bytree          : 1.0
• Importance Type           : split
• Learning Rate             : 0.1
• Max Depth                 : -1
• Min Child Samples         : 20
• Min Child Weight          : 0.001
• Min Split Gain            : 0.0
• N Estimators              : 100
• N Jobs                    : -1
• Num Leaves                : 31
• Objective                 : binary
• Random State              : 42
• Reg Alpha                 : 0.0
• Reg Lambda                : 0.0
• Subs

## 3.Buscas por hiperparamentros


In [10]:
# exploratorio
print("# Processo Iniciado em:", time.strftime("%H:%M:%S"))
param_dist_1 = {
    # Boosting
    'model__n_estimators': randint(200, 3000),
    'model__learning_rate': loguniform(0.005, 0.3),

    # Estrutura
    'model__max_depth': randint(3, 10),
    'model__num_leaves': randint(40, 200),

    # Regularização
    'model__min_child_samples': randint(10, 100),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.1, 0.9),
    'model__reg_alpha': uniform(0, 5),
    'model__reg_lambda': [0, 0.5, 1, 3, 5, 10]
}

cv_s = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
numero_itera =50

search_1 = RandomizedSearchCV(
    pipe_base,
    param_dist_1,
    n_iter=numero_itera,
    cv=cv_s,
    scoring=mtd_scoring,
    random_state=42,
    verbose=1
)

start = time.time()
search_1.fit(X_train, y_train)
end = time.time()


# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
best_1 = search_1.best_estimator_
y_pred_1=best_1.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_1,max_acc_1,y_probs_1=best_threshold2(best_1, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_1 = cross_val_score(best_1, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍  RandomizedSearchCV".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_1.mean():>15.4f} ± {cv_scores_1.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_1):>10.4f}")
print(f"   Otimizado:                 {max_acc_1:>10.4f} (threshold ={best_th_1:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_1):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_1):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_1.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))



# Processo Iniciado em: 11:54:20
Fitting 3 folds for each of 50 candidates, totalling 150 fits

#Finalizando a validação cruzada em: 12:58:29

                         📍  RandomizedSearchCV                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7280 ± 0.0014

✅ TEST SET
   Padrão (0.5):                  0.6867
   Otimizado:                     0.6852 (threshold = 0.520)
   ROC-AUC:                       0.7283
   Avg precision:                 0.8123
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample Bytree          : 0.2558816829190137
• Learning Rate             : 0.029540909142921727
• Max Depth                 : 9
• Min Child Samples         : 19
• N Estimators              : 1981
• Num Leaves                : 108
• Reg Alpha                 : 4.433401936490238
• Reg Lambda                : 3
• Subsample                 : 0.7498450458505885
--------------------------------------------------

#Processo finalizado em: 13:0

In [11]:
print("#Processo Iniciado em:", time.strftime("%H:%M:%S"))
def generate_refined_grid(best_params):
    """
    Gera um grid refinado automaticamente baseado nos melhores parâmetros
    do Random Search exploratório (LightGBM).
    """
    # ---------------------------
    # 1. n_estimators
    # ---------------------------
    n_best = best_params['model__n_estimators']
    n_min  = int(n_best * 0.8)
    n_max  = int(n_best * 1.2)
    # ---------------------------
    # 2. learning_rate
    # ---------------------------
    lr_best = best_params['model__learning_rate']
    lr_st   = lr_best * 0.7
    lr_end  = lr_best * 1.3
    # ---------------------------
    # 3. max_depth
    # ---------------------------
    depth_best = best_params['model__max_depth']
    depth_list = [max(1, depth_best - 1),
                  depth_best,depth_best + 1 ]
    # ---------------------------
    # 4. num_leaves
    # ---------------------------
    nl_best = best_params['model__num_leaves']
    nl_min  = max(8, int(nl_best * 0.7))
    nl_max  = int(nl_best * 1.3)

    # ---------------------------
    # 5. min_child_samples
    # ---------------------------
    mcs_best = best_params['model__min_child_samples']
    mcs_min  = max(5, int(mcs_best * 0.7))
    mcs_max  = int(mcs_best * 1.3)
    # ---------------------------
    # 6. subsample
    # ---------------------------
    ss_best = best_params['model__subsample']
    ss_st   = ss_best * 0.8
    ss_end  = min(1.0, ss_best + 0.2)
    # ---------------------------
    # 7. colsample_bytree
    # ---------------------------
    cb_best = best_params['model__colsample_bytree']
    cb_st   = cb_best * 0.8
    cb_end  = min(1.0, cb_best + 0.2)
    # ---------------------------
    # 8. regularização
    # ---------------------------
    alpha_best  = best_params['model__reg_alpha']
    lambda_best = best_params['model__reg_lambda']

    alpha_st = alpha_best * 0.7
    alpha_end = alpha_best + 0.5

    lambda_st = lambda_best * 0.7
    lambda_end = lambda_best + 0.5

    # ---------------------------
    # PRINTS DIAGNÓSTICOS
    # ---------------------------
    print(f"{'='*30} NOVO GRID DE REFINAMENTO (LightGBM) {'='*30}")
    print(f"🔹 n_estimators        : {n_min}  - {n_max}")
    print(f"🔹 learning_rate       : {lr_st:.5f} - {lr_end:.5f}")
    print(f"🔹 max_depth           : {depth_list}")
    print(f"🔹 num_leaves          : {nl_min} - {nl_max}")
    print(f"🔹 min_child_samples   : {mcs_min} - {mcs_max}")
    print(f"🔹 subsample           : {ss_st:.3f} - {ss_end:.3f}")
    print(f"🔹 colsample_bytree    : {cb_st:.3f} - {cb_end:.3f}")
    print(f"🔹 reg_alpha           : {alpha_st:.4f} - {alpha_end:.4f}")
    print(f"🔹 reg_lambda          : {lambda_st:.4f} - {lambda_end:.4f}")
    print(f"{'='*86}\n")

    # ---------------------------
    # GRID FINAL
    # ---------------------------
    refined_grid = {
        'model__n_estimators': randint(n_min, n_max + 1),
        'model__learning_rate': uniform(lr_st, lr_end - lr_st),
        'model__max_depth': depth_list,
        'model__num_leaves': randint(nl_min, nl_max + 1),
        'model__min_child_samples': randint(mcs_min, mcs_max + 1),
        'model__subsample': uniform(ss_st, ss_end - ss_st),
        'model__colsample_bytree': uniform(cb_st, cb_end - cb_st),
        'model__reg_alpha': uniform(alpha_st, alpha_end - alpha_st),
        'model__reg_lambda': uniform(lambda_st, lambda_end - lambda_st)
    }

    return refined_grid


# Aplicação automática
param_dist_2 = generate_refined_grid(search_1.best_params_)


numero_itera = 30
search_2 = RandomizedSearchCV(
    pipe_base, param_dist_2,
    n_iter=numero_itera, cv=cv_s, 
    scoring=mtd_scoring,
    random_state=42, verbose=1
)

start = time.time()
search_2.fit(X_train, y_train)
end = time.time()

# -----------------------------------------------------
# 1) Predição
# -----------------------------------------------------
best_2 = search_2.best_estimator_
y_pred_2=best_2.predict(X_test)

# -----------------------------------------------------
# 2) Otimização do threshold de decisão
# -----------------------------------------------------
best_th_2,max_acc_2,y_probs_2=best_threshold2(best_2, X_train, y_train,X_test,y_test)

print("\n#Finalizando a validação cruzada em:", time.strftime("%H:%M:%S"))

# -----------------------------------------------------
# 3) Desempenho em validação cruzada
# -----------------------------------------------------
cv_scores_2 = cross_val_score(best_2, X_train, y_train,
                                cv=cv_s,
                                scoring=mtd_scoring,
                                n_jobs=-1) #90s

print(f"\n{'='*70}")
print(f" 📍  RandomizedSearchCV".center(70))
print(f"{'='*70}")

# -----------------------------------------------------
# 4) Avaliação por validação cruzada (Treino)
# -----------------------------------------------------
print("📊 CROSS-VALIDATION")
print(f"   Média {mtd_scoring}:       {cv_scores_2.mean():>15.4f} ± {cv_scores_2.std():.4f}")

# -----------------------------------------------------
# 5) Avaliação no conjunto de teste
# -----------------------------------------------------
print(f"\n✅ TEST SET")
print(f"   Padrão (0.5):              {accuracy_score(y_test, y_pred_2):>10.4f}")
print(f"   Otimizado:                 {max_acc_2:>10.4f} (threshold ={best_th_2:>6.3f})")
print(f"   ROC-AUC:                   {roc_auc_score(y_test, y_probs_2):>10.4f}")
print(f"   Avg precision:             {average_precision_score(y_test, y_probs_2):>10.4f}")

#-----------------------------------------------------
# 6) Hiperparametros utilizados
#-----------------------------------------------------
print(f"{'='*70}")
print_hiper(search_2.best_params_)
print("\n#Processo finalizado em:", time.strftime("%H:%M:%S"))

#Processo Iniciado em: 13:29:13
============================== NOVO GRID DE REFINAMENTO (LightGBM) ==============================
🔹 n_estimators        : 1584  - 2377
🔹 learning_rate       : 0.02068 - 0.03840
🔹 max_depth           : [8, 9, 10]
🔹 num_leaves          : 75 - 140
🔹 min_child_samples   : 13 - 24
🔹 subsample           : 0.600 - 0.950
🔹 colsample_bytree    : 0.205 - 0.456
🔹 reg_alpha           : 3.1034 - 4.9334
🔹 reg_lambda          : 2.1000 - 3.5000

Fitting 3 folds for each of 30 candidates, totalling 90 fits

#Finalizando a validação cruzada em: 14:38:06

                         📍  RandomizedSearchCV                        
📊 CROSS-VALIDATION
   Média roc_auc:                0.7284 ± 0.0013

✅ TEST SET
   Padrão (0.5):                  0.6862
   Otimizado:                     0.6863 (threshold = 0.510)
   ROC-AUC:                       0.7284
   Avg precision:                 0.8124
🎯 Melhores Hiperparâmetros
--------------------------------------------------
• Colsample 

In [6]:
# end_final = time.time()
# print(f"Tempo final: {(end_final-start_inicial)/60:.2f} min")


Tempo final: 69.58 min


In [12]:
# #Salvar Hiperparametros joblib

joblib.dump(search_2, 'search2_refine.'+mtd_scoring+'_v1.2.joblib')
joblib.dump(search_1, 'search2_randsearch.'+mtd_scoring+'_v1.2.joblib')

joblib.dump(search_1.best_estimator_, 'modelo_LGBM_final_randsearch.'+mtd_scoring+'_v1.2.joblib')
joblib.dump(search_2.best_estimator_, 'modelo_LGBM_final_refine.'+mtd_scoring+'_v1.2.joblib')

['modelo_LGBM_final_refine.roc_auc_v1.2.joblib']